In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import logging
import os
from pathlib import Path
import anndata2ri
from bokeh.models import TabPanel, Tabs, ColorBar
from bokeh.plotting import show, output_file
from scipy.stats import median_abs_deviation
import seaborn as sns
import scrublet as scr
import seaborn as sns
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from utils import interactive_embedding, gridlayout
import anndata as ad
from scipy.sparse import csr_matrix, issparse
from bokeh.plotting import show, output_file, output_notebook
from bokeh.layouts import row, grid
import warnings
import numpy as np
import scib

warnings.filterwarnings("ignore")
anndata2ri.activate()
%load_ext rpy2.ipython

# rcb.logger.setLevel(logging.ERROR)

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False, figsize=(8, 5))

pd.set_option("display.max_columns", None)
sc.settings.verbosity = 1
output_notebook()
%matplotlib inline

In [1]:
BATCH = "batch_1"

In [ ]:
batch_map = {"batch_1": "TD005881", "batch_1": "TD005881"}

In [ ]:
project_path = Path(
    "/Users/nicolebussola/Desktop/Projects/LBP_MountSinai/LBP_brain_blood_pairs/data/TD005881/"
)
adata = sc.read(project_path / "blood_4000cell_ranger_harmony_clustered.h5ad")

In [ ]:
meta_df = pd.read_csv(project_path / "Batch2_celllevel_metadata.tsv",sep="\t")
meta_df['donor'] = meta_df['donor'].apply(lambda x: ('-').join((x.split('-')[0], x.split('-')[1][1:])))
meta_df.head(n=3)
# pd.unique(meta_df['donor'])

In [ ]:
age_dict = meta_df.groupby('donor')['age'].agg(lambda x: x.unique()[0]).to_dict()
sex_dict = meta_df.groupby('donor')['sex'].agg(lambda x: x.unique()[0]).to_dict()
diag_dict = meta_df.groupby('donor')['diagnosis'].agg(lambda x: x.unique()[0]).to_dict()


In [ ]:
adata.obs = adata.obs[['side', 'pt', 'log1p_n_genes_by_counts','log1p_total_counts','pct_counts_in_top_20_genes', "pct_counts_mt", 'pct_counts_ribo', 'leiden_res1_25', 'leiden_res1_75']]
adata.obs['age'] = adata.obs['pt'].map(age_dict).astype(int)
adata.obs['sex'] = adata.obs['pt'].map(sex_dict)
adata.obs['diagnosis'] = adata.obs['pt'].map(diag_dict)

In [ ]:
# adata.layers["counts"] = adata.X.copy()
# adata.X = adata.layers["scran_normalization"].copy()

In [ ]:
# [i for i in adata.obs.index if 'AAACCCAAGCGTATAA' in i], [i for i in meta_df['cell'] if 'AAACCCAAGCGTATAA' in i]

In [ ]:
age_dict

In [ ]:
labels = [
    "pt",
    "side",
    "age",
    "sex",
    "diagnosis",
    "log1p_total_counts",
    "log1p_n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "leiden_res1_75"
]

sc.pl.umap(adata, color=labels, wspace=1)
gridlayout(
    labels,
    adata,
    width=700,
    height=500,
    ncols=2,
    # labels_loc="on_data",
)

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=['leiden_res1_25', 'leiden_res1_75'], wspace=1)


In [ ]:
marker_genes = pd.read_pickle("marker_genes.pkl")

In [ ]:
marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
B_plasma_cts = [
    "Naive CD20+ B",
    "B1 B",
    "Transitional B",
    "Plasma cells",
    "Plasmablast",
]

T_cts = [
    "CD4+ T activated",
    "CD4+ T naive",
    "CD8+ T",
    "T activation",
    "T naive",
]

Mono = ["CD14+ Mono", "CD16+ Mono"]

NK = ["NK"]

DC = [
    "pDC",
    "cDC1",
    "cDC2",
]

# for ct in Mono:
#     print(f"{ct.upper()}:")  # print cell subtype name
#     sc.pl.umap(
#         adata,
#         color=marker_genes_in_data[ct],
#         vmin=0,
#         vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
#         sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
#         frameon=False,
#         cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
#     )
#     print("\n\n\n")

In [ ]:
B_plasma_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in B_plasma_cts
}

T_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in T_cts
}

Mono_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in Mono
}

NK_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in NK
}

DC_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in DC
}

In [ ]:
marker_genes.keys(), marker_genes[ 'NK']

In [ ]:
# sc.pl.umap(adata, color="leiden_res1_5", legend_loc="on data")

In [ ]:
marker_genes["pDC"], DC_markers

In [ ]:
sc.pl.dotplot(
    adata,
    groupby="leiden_res1_75",
    var_names=DC_markers, #[g for g in marker_genes["CD14+ Mono"] if g in adata.var.index],
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
)

In [ ]:
cl_annotation_bp_175 = {
    "7": "Naive CD20+ B",
    "18": "Naive CD20+ B",
    "25": "Naive CD20+ B",
    "28": "Naive CD20+ B",
    "32": "Plasma cells?",  
    "34": "CD8+ T or Monocytes",
    "6": "CD8+ T",
    "8": "CD4+ T",
    "4": "Cytotoxic T",
    "9": "Cytotoxic T",
    "24": "Cytotoxic T",
    "19": "Cytotoxic T",
    "32": "CD14+ Mono",
    "26": "CD14+ Mono",
    "33": "CD14+ Mono",
    "13": "CD14+ Mono",
    "14": "CD14+ Mono",
    "10": "CD16+ Mono",
    "2": "CD14+ Mono",
    "3": "CD14+ Mono",
    "9": "NK",
    "30": "pDC",
    # Markers found using the scTypeDB dataset
}

cl_annotation_bp_15 = {}

In [ ]:
# adata.obs["manual_celltype_annotation_bp_15"] = adata.obs.leiden_res1_5.map(
#     cl_annotation_bp_15
# )
adata.obs["manual_celltype_annotation_bp_175"] = adata.obs.leiden_res1_75.map(
    cl_annotation_bp_175
)

In [ ]:
sc.pl.umap(
    adata,
    color=["manual_celltype_annotation_bp_175"],
    legend_loc="on data",
)
adata.uns["manual_celltype_annotation_bp_175_colors"] += ["#AC24FC"]
# adata.uns["manual_celltype_annotation_bp_15_colors"] += ["#AC24FC"]
# adata.obs["manual_celltype_annotation_bp_15"] = adata.obs[
#     "manual_celltype_annotation_bp_15"
# ].cat.add_categories("NA")

adata.obs["manual_celltype_annotation_bp_175"] = adata.obs[
    "manual_celltype_annotation_bp_175"
].cat.add_categories("NA")
adata.obs = adata.obs.fillna("NA")

In [ ]:
gridlayout(
    [
        "pt",
        "side",
        "leiden_res1_5",
        "leiden_res1_75",
        "manual_celltype_annotation_bp_175",
    ],
    adata,
    width=700,
    height=500,
    ncols=2,
    labels_loc="on_data",
)

In [ ]:
marker_genes_sctype_df = pd.read_csv(
    "ScTypeDB_full.csv", sep=";", on_bad_lines="skip"
)

marker_genes_sctype_df = marker_genes_sctype_df[
    marker_genes_sctype_df["tissueType"] == "Immune system"
]
marker_genes_sctype_df["geneSymbolmore1"] = marker_genes_sctype_df[
    "geneSymbolmore1"
].apply(lambda x: x.split(","))

marker_genes_sctype_df["geneSymbolmore2"] = marker_genes_sctype_df[
    "geneSymbolmore2"
].apply(lambda x: str(x).split(","))
marker_genes_sctype_df["geneSymbolmore12"] = (
    marker_genes_sctype_df["geneSymbolmore1"]
    + marker_genes_sctype_df["geneSymbolmore2"]
)

In [ ]:
marker_genes_sctype = (
    marker_genes_sctype_df.groupby("shortName")["geneSymbolmore12"]
    .apply(lambda x: [item for item in x])
    .to_dict()
)
marker_genes_sctype.keys()

In [ ]:
cell_markers = {
    ct: [m for m in ct_markers[0] if m in adata.var.index]
    for ct, ct_markers in marker_genes_sctype.items()
    if ct
    in ['CD4+ NKT', 'CD8+ NKT', 'Memory CD4+ T', 'Memory CD8+ T','Naive CD4+ T', 'Naive CD8+ T'] # ["Platelets", "Basophils", "Eosinophils", "Es", "Gs", "HSC/MPP", "ISG+", "Mast",  'Pre-B', 'Pro-B', 'mDCs', 'pDCs', 'γδ-T']
}

In [ ]:
type = 'mDCs'
len(marker_genes_sctype[type][0])

In [ ]:
sc.pl.dotplot(
    adata,
    groupby="leiden_res1_75",
    var_names= [g for g in marker_genes_sctype[type][0] if g in adata.var.index],
    standard_scale="var",
    swap_axes=False,  # standard scale: normalize each gene to range from 0 to 1
)

In [ ]:
cl_annotation_bp_175["31"] = "Platelets"
cl_annotation_bp_175["22"]="Th17-Tregs"


In [ ]:
# adata.obs["manual_celltype_annotation_bp_175"] = adata.obs.leiden_res1_75.map(
#     cl_annotation_bp_175
# )
# sc.pl.umap(adata, color="manual_celltype_annotation_bp_175", legend_loc="on data")
# adata.uns["manual_celltype_annotation_bp_175_colors"] += ["#AC24FC"]
# adata.obs["manual_celltype_annotation_bp_175"] = adata.obs[
#     "manual_celltype_annotation_bp_175"
# ].cat.add_categories("NA")
# adata.obs = adata.obs.fillna("NA")

# gridlayout(
#     ["leiden_res1_75", "manual_celltype_annotation_bp_175"],
#     adata,
#     width=900,
#     height=700,
#     labels_loc="on_data",
# )

In [ ]:
adata.uns["log1p"]["base"] = None

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby="leiden_res1_75", method="wilcoxon", key_added="dea_leiden_1_75"
)
sc.tl.filter_rank_genes_groups(
    adata,
    min_in_group_fraction=0.2,
    max_out_group_fraction=0.2,
    key="dea_leiden_1_75",
    key_added="dea_leiden_1_75_filtered",
)

In [ ]:
# sc.pl.rank_genes_groups_dotplot(
#     adata,
#         groupby="leiden_res1_5",
#     n_genes=10,
#         key=`"dea_leiden_1_5_filtered",
#     values_to_plot="logfoldchanges", cmap='bwr',
#     colorbar_title='log fold change'
# # )

In [ ]:
# gridlayout(
#     ["leiden_res1_5", "manual_celltype_annotation_bp_15", "leiden_res1_75", "manual_celltype_annotation_bp_175"],
#     adata,
#     width=900,
#     height=700,
#     ncols=2,
#     labels_loc="on_data",
# )

In [ ]:
cl_annotation_bp_175
sorted_anno = {str(key): value for key, value in sorted(cl_annotation_bp_175.items(), key=lambda item: int(item[0]))}
sorted_anno

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden_res1_75",
    standard_scale="var",
    n_genes=5,
    key="dea_leiden_1_75_filtered",
)

In [ ]:
# sc.pl.rank_genes_groups_dotplot(
#     adata,
#     groupby="leiden_res1_5",
#     standard_scale="var",
#     n_genes=10,
#     key="dea_leiden_1_5_filtered",
# )

In [ ]:
cl_annotation_bp_175["16"] = "Mono+DC"
cl_annotation_bp_175["4"] = "NK"

In [ ]:
adata.obs["manual_celltype_annotation_bp_175"] = adata.obs.leiden_res1_75.map(
    cl_annotation_bp_175
)
sc.pl.umap(adata, color=["manual_celltype_annotation_bp_175"], legend_loc="on data")
# adata.uns["manual_celltype_annotation_bp_15_colors"] += ["#AC24FC"]
adata.uns["manual_celltype_annotation_bp_175_colors"] += ["#AC24FC"]

adata.obs["manual_celltype_annotation_bp_175"] = adata.obs[
    "manual_celltype_annotation_bp_175"
].cat.add_categories("NA")
adata.obs = adata.obs.fillna("NA")
# gridlayout(
#     ["leiden_res1_75", "manual_celltype_annotation_bp_175"],
#     adata,
#     width=900,
#     height=700,
#     ncols=2,
#     labels_loc="on_data",
# )

## CellTypist

In [ ]:
print(adata.layers["counts"])

In [ ]:
adata_celltypist = adata.copy()  # make a copy of our adata
adata_celltypist.X = adata.layers["counts"] # set adata.X to raw counts
sc.pp.normalize_per_cell(
    adata_celltypist, counts_per_cell_after=10**4
)  # normalize to 10,000 counts per cell
sc.pp.log1p(adata_celltypist)  # log-transform
# make .X dense instead of sparse, for compatibility with celltypist:
adata_celltypist.X = np.round(adata_celltypist.X, 5).toarray()

In [ ]:
np.max(adata_celltypist.X ),  np.log1p(10000)

In [ ]:
from celltypist import models
import celltypist

models.download_models(
    force_update=True, model=["Immune_All_Low.pkl", "Immune_All_High.pkl"]
)

In [ ]:
# models.models_description(on_the_fly = True)


In [ ]:
model_low = models.Model.load(model="Immune_All_Low.pkl")
model_high = models.Model.load(model="Immune_All_High.pkl")

In [ ]:
predictions_high = celltypist.annotate(
    adata_celltypist, model=model_high, majority_voting=True
)

In [ ]:
predictions_high_adata = predictions_high.to_adata()

In [ ]:
adata.obs["celltypist_cell_label_coarse"] = predictions_high_adata.obs.loc[
    adata.obs.index, "majority_voting"
]
adata.obs["celltypist_conf_score_coarse"] = predictions_high_adata.obs.loc[
    adata.obs.index, "conf_score"
]

In [ ]:
predictions_low = celltypist.annotate(
    adata_celltypist, model=model_low, majority_voting=True
)
predictions_low_adata = predictions_low.to_adata()

In [ ]:
adata.obs["celltypist_cell_label_fine"] = predictions_low_adata.obs.loc[
    adata.obs.index, "majority_voting"
]
adata.obs["celltypist_conf_score_fine"] = predictions_low_adata.obs.loc[
    adata.obs.index, "conf_score"
]

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "celltypist_cell_label_coarse",
        "celltypist_conf_score_coarse",
        "celltypist_cell_label_fine",
        "celltypist_conf_score_fine",
    ],
    frameon=False,
    sort_order=False,
    wspace=1,
)

In [ ]:
gridlayout(
    [
        "celltypist_cell_label_coarse",
        "celltypist_conf_score_coarse",
        "celltypist_cell_label_fine",
        "celltypist_conf_score_fine",
        "leiden_res1_75",
        "manual_celltype_annotation_bp_175",
    ],
    adata,
    width=900,
    height=700,
    ncols=2,
    labels_loc="on_data",
)

In [ ]:
label_scarches = pd.read_csv(project_path / "blood_scarches_annotation.csv")
label_scarches = label_scarches.rename({'Unnamed: 0':'cell'}, axis=1)
label_scarches

In [ ]:
(adata.obs.index == label_scarches['cell']).all()

In [ ]:
adata.obs['cell']=adata.obs.index
adata.obs['label_scarches'] = list(pd.merge(adata.obs, label_scarches, on='cell')['transf_cell_type_certain'])


In [ ]:
print([i for i in pd.unique(adata.obs['label_scarches'])])

types_of_interst = ['CD14+ Mono', 'Plasma', 'cDC2', 'pDC', 'CD16+ Mono', 'HSC', 'CD4+ T activated', 'B1 B', 'NK', 'CD4+ T naive', 'Naive CD20+ B', 'CD8+ T activated', 'CD8+ T naive', 'Other T', 'MK/E prog',]
final_anno = adata.obs[adata.obs['label_scarches'].isin(types_of_interst)]
final_anno['consensus_label'] = final_anno['label_scarches']


In [ ]:
T_reg_cells = adata.obs[(adata.obs['manual_celltype_annotation_bp_175']=="Th17-Tregs")& (adata.obs['label_scarches']=="Unknown")]
T_reg_cells['consensus_label']='Tregs'
T_reg_cells.shape

In [ ]:
pd.concat([final_anno, T_reg_cells])

In [ ]:
adata_annotated = adata[adata.obs.index.isin(list(pd.concat([final_anno, T_reg_cells]).index))]
adata_annotated

In [ ]:
from collections import Counter
adata_annotated.obs[adata_annotated.obs['label_scarches']=="Unknown"]

adata_annotated.obs['label_scarches'] = adata_annotated.obs['label_scarches'].replace('Unknown', 'Treg')
adata_annotated.obs

In [ ]:
sc.pl.umap(
    adata_annotated,
    color='label_scarches',
    frameon=False,
    sort_order=False,
    wspace=1,
)

In [ ]:
# pd.concat([final_anno, T_reg_cells]).to_csv(project_path / "blood_consensus_annotation.csv")

In [ ]:
adata_annotated.obs

In [ ]:
# adata_annotated.write(project_path / "blood_consensus_annotated.h5ad")
adata_annotated

In [ ]:
 labels=   [        
         "pt",
        "side",
        "age",
        "sex",
        "diagnosis",
        "label_scarches",
        "pct_counts_mt",
        "pct_counts_ribo",
        "log1p_total_counts"
    ]
sc.pl.umap(
    adata_annotated,
    color=labels,
    frameon=False,
    sort_order=False,
    wspace=1,
)
gridlayout(
    ["label_scarches"],
    adata_annotated,
    width=1000,
    height=900,
    ncols=1,
    labels_loc="on_data",
    fname = project_path / "blood_harmony_annotations_consensus2.html"
)

In [ ]:
# adata.obs.to_csv(project_path / "blood_4000harmony_manualCelltype_annotation.csv")

In [ ]:
# https://www.pnas.org/doi/pdf/10.1073/pnas.2002476117 for SMIM25 and CD16+ markers

In [ ]:
[i for i in adata.var_names if "CD8" in i]

In [ ]:
sc.pl.umap(
    adata,
    color=["CD8A", "HBB"],
    vmin=0,
    vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
    sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
    frameon=False,
    cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
)

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(20, 15))
# fig, ax = plt.subplots(figsize=(20, 15))

# sc.pl.umap(
#     adata,
#     color="CLEC9A",
#     vmin=0,
#     vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
#     sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
#     frameon=False,
#     cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
#     ax=ax,
#     size=8,
# )

In [ ]:
adata.obs["manual_celltype_annotation_bp_15"] = adata.obs.leiden_res1_5.map(
    cl_annotation_bp_15
)
adata.obs["manual_celltype_annotation_bp_175"] = adata.obs.leiden_res1_75.map(
    cl_annotation_bp_175
)

In [ ]:
adata.obs["manual_celltype_annotation_bp_15"] = adata.obs.leiden_res1_5.map(
    cl_annotation_bp_15
)
adata.obs["manual_celltype_annotation_bp_175"] = adata.obs.leiden_res1_75.map(
    cl_annotation_bp_175
)
sc.pl.umap(
    adata,
    color=["manual_celltype_annotation_bp_15", "manual_celltype_annotation_bp_175"],
    legend_loc="on data",
)
adata.uns["manual_celltype_annotation_bp_15_colors"] += ["#AC24FC"]
adata.uns["manual_celltype_annotation_bp_175_colors"] += ["#AC24FC"]

adata.obs["manual_celltype_annotation_bp_15"] = adata.obs[
    "manual_celltype_annotation_bp_15"
].cat.add_categories("NA")
adata.obs["manual_celltype_annotation_bp_175"] = adata.obs[
    "manual_celltype_annotation_bp_175"
].cat.add_categories("NA")
adata.obs = adata.obs.fillna("NA")

In [ ]:
gridlayout(
    [
        "pt",
        "side",
        "log1p_total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
    
        "leiden_res1_75",
        "manual_celltype_annotation_bp_175",
        "celltypist_cell_label_coarse",
        "celltypist_conf_score_coarse",
        "celltypist_cell_label_fine",
        "celltypist_conf_score_fine",
    ],
    adata,
    width=900,
    height=700,
    ncols=4,
    labels_loc="on_data",
    fname=project_path /  "blood_4000chem_harmony_clustered_annotated.html",
)

In [ ]:
# sc.pl.dendrogram(adata, groupby="celltypist_cell_label_fine")

In [ ]:
# adata.write(project_path / "data" / "blood_4000chem_harmony_clustered_annotated.h5ad")
adata.obs.to_csv(
    project_path / "blood_4000chem_harmony_clustered_annotated.csv"
)

In [ ]:
%%R
library(scAnnotate)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="cell_ranger",
    batch_key="pt",
)

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg

In [ ]:
adata

In [ ]:
adata_df = pd.DataFrame.sparse.from_spmatrix(data=csr_matrix(adata.layers['log1p_norm']), columns=adata.var_names, index=adata.obs_names)
# adata_df.to_csv(project_path / "prova_blood.csv")
adata_df

In [ ]:
# %%R -i adata_df
# predict_label=scAnnotate(train=data(pbmc1),
#                          test=csr_matrix(adata.layers['log1p_norm']),
#                          distribution="normal",
#                          correction ="auto",
#                          screening = "wilcox",
#                          threshold=0,
#                          lognormalized=TRUE)

# ?data(pbmc1)

In [ ]:
# eva_cal(prediction = predict_label,cell_label = pbmc2[,1])


## ScARches

In [ ]:
adata_all = sc.read(project_path / "blood_filtered_normalized.h5ad")

In [ ]:
adata_to_map = adata_all.copy()
for layer in list(adata_to_map.layers.keys()):
    if layer != "counts":
        del adata_to_map.layers[layer]
adata_to_map.X = adata_to_map.layers["counts"]

In [ ]:
reference_model_features = pd.read_csv(
    "https://figshare.com/ndownloader/files/41436645", index_col=0
)

In [ ]:
adata_to_map.var["gene_names"] = adata_to_map.var.index
print("Total number of genes needed for mapping:", reference_model_features.shape[0])

In [ ]:
print(
    "Number of genes found in query dataset:",
    adata_to_map.var.index.isin(reference_model_features.index).sum(),
)

In [ ]:
missing_genes = [
    gene_id
    for gene_id in reference_model_features.index
    if gene_id not in adata_to_map.var.index
]
missing_gene_adata = sc.AnnData(
    X=csr_matrix(np.zeros(shape=(adata.n_obs, len(missing_genes))), dtype="float32"),
    obs=adata.obs.iloc[:, :1],
    var=reference_model_features.loc[missing_genes, :],
)
missing_gene_adata.layers["counts"] = missing_gene_adata.X

In [ ]:
if "PCs" in adata_to_map.varm.keys():
    del adata_to_map.varm["PCs"]

adata_to_map_augmented = sc.concat(
    [adata_to_map, missing_gene_adata],
    axis=1,
    join="outer",
    index_unique=None,
    merge="unique",
)


In [ ]:
adata_to_map_augmented = adata_to_map_augmented[
    :, reference_model_features.index
].copy()

(adata_to_map_augmented.var.index == reference_model_features.index).all()

In [ ]:
adata_to_map_augmented.var["gene_ids"] = adata_to_map_augmented.var.index
adata_to_map_augmented.var.set_index("gene_names", inplace=True)

In [ ]:
import scarches as sca
import urllib

# loading model.pt from figshare
if not os.path.exists("./reference_model"):
    os.mkdir("./reference_model")
elif not os.path.exists("./reference_model/model.pt"):
    urllib.request.urlretrieve(
        "https://figshare.com/ndownloader/files/41436648",
        filename="reference_model/model.pt",
    )

In [ ]:
adata_to_map_augmented.obs["batch"] = adata_to_map_augmented.obs["pt"]

In [ ]:
scarches_model = sca.models.SCVI.load_query_data(
    adata=adata_to_map_augmented,
    reference_model="./reference_model",
    freeze_dropout=True,
)

In [ ]:
scarches_model.train(max_epochs=500, plan_kwargs=dict(weight_decay=0.0))

In [ ]:
adata

In [ ]:
adata_all

In [ ]:
adata_all.obsm["X_scVI"] = scarches_model.get_latent_representation()
sc.pp.neighbors(adata_all, use_rep="X_scVI")
sc.tl.umap(adata_all)

In [ ]:
sc.pl.umap(
    adata,
    color=["IGHD", "IGHM", "PRDM1"],
    vmin=0,
    vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
    sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
    frameon=False,
    cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
)

In [ ]:
ref_emb = sc.read(
    filename="reference_embedding.h5ad",
    backup_url="https://figshare.com/ndownloader/files/41376264",
)

In [ ]:
ref_emb.obs["reference_or_query"] = "reference"

In [ ]:
adata_emb = sc.AnnData(X=adata_all.obsm["X_scVI"], obs=adata_all.obs)
adata_emb.obs["reference_or_query"] = "query"

In [ ]:
emb_ref_query = sc.concat(
    [ref_emb, adata_emb],
    axis=0,
    join="outer",
    index_unique=None,
    merge="unique",
)

In [ ]:
sc.pp.neighbors(emb_ref_query)
sc.tl.umap(emb_ref_query)
sc.pl.umap(
    emb_ref_query,
    color=["reference_or_query"],
    sort_order=False,
    frameon=False,
)

In [ ]:
sc.pl.umap(
    emb_ref_query,
    color=["cell_type"],
    sort_order=False,
    frameon=False,
    legend_loc="on data",
    legend_fontsize=10,
    na_color="black",
)

In [ ]:
knn_transformer = sca.utils.knn.weighted_knn_trainer(
    train_adata=ref_emb,
    train_adata_emb="X",  # location of our joint embedding
    n_neighbors=15,
)

In [ ]:
labels, uncert = sca.utils.knn.weighted_knn_transfer(
    query_adata=adata_emb,
    query_adata_emb="X",  # location of our embedding, query_adata.X in this case
    label_keys="cell_type",  # (start of) obs column name(s) for which to transfer labels
    knn_model=knn_transformer,
    ref_adata_obs=ref_emb.obs,
)

In [ ]:
adata_emb.obs["transf_cell_type"] = labels.loc[adata_emb.obs.index, "cell_type"]
adata_emb.obs["transf_cell_type_unc"] = uncert.loc[adata_emb.obs.index, "cell_type"]

In [ ]:
adata_all.obs.loc[adata_emb.obs.index, "transf_cell_type"] = adata_emb.obs[
    "transf_cell_type"
]
adata_all.obs.loc[adata_emb.obs.index, "transf_cell_type_unc"] = adata_emb.obs[
    "transf_cell_type_unc"
]

In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
sc.pl.umap(adata_all, color="transf_cell_type", frameon=False)

In [ ]:
sc.pl.umap(adata_all, color="transf_cell_type_unc", frameon=False)


In [ ]:
adata_all.obs

In [ ]:
adata_all.obs['transf_cell_type_unc']=adata_all.obs['transf_cell_type_unc'].astype(np.float32)
ct_order

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(8, 3))
ct_order = (
    adata_all.obs.groupby("transf_cell_type")
    .agg({"transf_cell_type_unc": "median"})
    .sort_values(by="transf_cell_type_unc", ascending=False)
)
sns.boxplot(
    data=adata_all.obs,
    x="transf_cell_type",
    y="transf_cell_type_unc",
    color="grey",
    ax=ax,
    order=ct_order.index,
)
ax.tick_params(rotation=90, axis="x")

In [ ]:
adata_all.obs["transf_cell_type_certain"] = adata_all.obs.transf_cell_type.tolist()
adata_all.obs.loc[
    adata_all.obs.transf_cell_type_unc > 0.2, "transf_cell_type_certain"
] = "Unknown"

In [ ]:
sc.pl.umap(adata_all, color="transf_cell_type_certain", frameon=False)

In [ ]:
adata_all.obs.to_csv(project_path / "blood_scarches_annotation.csv")